# 12 · E4: FP8 base model vs bf16 (Steps 4.1–4.2, D-057)

**Runtime: L4 GPU** (Runtime → Change runtime type → **L4 GPU**). FP8 needs an L4 (or newer); a T4 cannot do it.

The model is stored with 8-bit weights instead of 16-bit. vLLM converts it while loading, so there is no separate preparation step. The E5 adapter stays in 16-bit and is attached on top, unchanged.

1. **Speed and memory check (~15 min):** the same 1,000 answering questions, once with bf16 and once with FP8, in the same session. We record photos per second, the weights' memory, and how much room is left for requests in progress.
2. **Full accuracy check (~40 min):** all three test sets with FP8.
3. **Compare** with bf16 (the E5 numbers) on the same examples.

No training, so there is no W&B key and no resume. If Colab disconnects, **Run all** again: finished tasks are skipped.

Every command runs through `sh()`, which stops with an error if the command fails.

## 1 · Settings, Drive, code

In [ ]:
import json
import os
import re
import subprocess

from google.colab import drive

REPO_URL = "https://github.com/Vishwapujara/vizwiz-assistant.git"
BRANCH = "main"
REPO_DIR = "/content/vizwiz-assistant"
DRIVE_ROOT = "/content/drive/MyDrive/vizwiz"
DATA = f"{DRIVE_ROOT}/datasets"
LOCAL = "/content/data"
RUNS = f"{DRIVE_ROOT}/runs"
VENV = "/content/venv"
PY = f"{VENV}/bin/python"
ADAPTER = f"{DRIVE_ROOT}/checkpoints/e5-multi-lora/adapter"
RUN_ID = "e4-fp8-e5"
OUT = f"{RUNS}/{RUN_ID}"
BENCH = f"{RUNS}/e4-speed"
TASKS = ("answer", "answerability", "quality")
# vLLM start-up lines that report memory
MEM_PATTERNS = (
    "Model loading took",
    "Available KV cache memory",
    "GPU KV cache size",
    "Maximum concurrency",
)


def sh(cmd, check=True, **env):
    """Run a shell command, stream its output, raise if it fails; returns the output lines."""
    proc = subprocess.Popen(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env={**os.environ, **env},
    )
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    code = proc.wait()
    if check and code:
        raise RuntimeError(f"Command failed (exit {code}). Stop here and paste the output above.")
    return lines


def memory_lines(lines):
    return [
        re.sub(r"^.*?\] ", "", ln).strip() for ln in lines if any(p in ln for p in MEM_PATTERNS)
    ]


os.environ["HF_HOME"] = "/content/hf_cache"
drive.mount("/content/drive")
assert os.path.exists(f"{ADAPTER}/adapter_config.json"), "E5 adapter not found on Drive"
sh(
    f"if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}; else git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}; fi"
)
sh(f"git -C {REPO_DIR} log --oneline -1")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")

**Paste back:** the commit line and the GPU line (must say NVIDIA L4).

## 2 · Pinned environment (~3–5 min)

In [ ]:
%%time
sh("pip install -q uv")
sh(f"uv venv -q --clear --python 3.13 {VENV}")
sh(
    f"uv pip install -q --python {PY} -e {REPO_DIR} -r {REPO_DIR}/requirements/colab-gpu.txt -c {REPO_DIR}/requirements/colab-gpu.lock.txt"
)
sh(f"{PY} -m vizwiz.compat versions")

## 3 · Unzip test photos (~2 min)

In [ ]:
%%time
import hashlib

expected = json.load(open(f"{REPO_DIR}/results/splits/files.json"))
for name in ("vqa_test.jsonl", "quality_test.jsonl"):
    digest = hashlib.sha256(open(f"{DATA}/processed/{name}", "rb").read()).hexdigest()
    assert digest == expected[name]["sha256"], f"{name} differs from the committed hash"
    print(f"{name}: matches committed hash")
sh(f"{PY} -m vizwiz.data.download extract --root {DATA} --dest {LOCAL} --splits val")

## 4 · Speed and memory: bf16 vs FP8 (~15 min)
The same 1,000 answering questions (fixed sample), the same batch size, one engine start each. The first start downloads the model (~9 GB). If the FP8 start fails, stop and paste the error.

In [ ]:
%%time
speed = {}
for q in ("none", "fp8"):
    out = f"{BENCH}/{q}"
    lines = sh(
        f"{PY} -m vizwiz.eval.run_eval --records {DATA}/processed --images {LOCAL}/images --split test --tasks answer --limit 1000 --adapter {ADAPTER} --quantization {q} --run-id e4-speed-{q} --out {out}"
    )
    m = json.load(open(f"{out}/metrics.json"))
    t = m["tasks"]["answer"]
    speed[q] = {
        "examples_per_s": t["timing"]["examples_per_s"],
        "engine_load_s": m["engine_load_s"],
        "accuracy_1000": t["metrics"]["vqa_accuracy"],
        "memory": memory_lines(lines),
    }
json.dump(speed, open(f"{BENCH}/speed_memory.json", "w"), indent=2)
for q, s in speed.items():
    print(f"SPEED {q}:", {k: v for k, v in s.items() if k != "memory"})
    for ln in s["memory"]:
        print(f"  MEM {q}: {ln}")

**Paste back:** the two `SPEED` lines and all `MEM` lines.

## 5 · Full accuracy check with FP8 (~40 min)
All three test sets. Finished tasks are kept and skipped on a rerun.

In [ ]:
%%time
done = (
    set(json.load(open(f"{OUT}/metrics.json"))["tasks"])
    if os.path.exists(f"{OUT}/metrics.json")
    else set()
)
todo = [t for t in TASKS if t not in done]
if not todo:
    print("evaluation already done: skipped")
else:
    sh(
        f"{PY} -m vizwiz.eval.run_eval --records {DATA}/processed --images {LOCAL}/images --split test --tasks {' '.join(todo)} --adapter {ADAPTER} --quantization fp8 --run-id {RUN_ID} --out {OUT}"
    )

**Paste back:** the three result lines (`[answer] vqa_accuracy`, `[answerability] average_precision`, `[quality] macro_f1`).

## 6 · Compare: bf16 vs FP8 (~2 min)
Paired on the same examples; the reference is E5 in bf16, so the FP8 difference reads "FP8 minus bf16". The reading rule was fixed before the run (D-057).

In [ ]:
for t in TASKS:
    sh(
        f"{PY} -m vizwiz.eval.compare --records {DATA}/processed --split test --task {t} --runs {RUNS}/e1-baseline {RUNS}/e5-multi-lora {OUT} --names e1 e5_bf16 e5_fp8 --reference e5_bf16 --report {OUT}/compare_{t}.json > /dev/null"
    )
    c = json.load(open(f"{OUT}/compare_{t}.json"))
    print(
        f"COMPARE {t} {c['metric']} n={c['n']}:",
        {k: (r["value"], r["diff_vs_reference"], r["diff_ci95"]) for k, r in c["runs"].items()},
    )
m = json.load(open(f"{OUT}/metrics.json"))["tasks"]
print(
    "FP8 answerability invalid %:",
    m["answerability"]["metrics"]["invalid_output_pct"],
    " quality JSON valid %:",
    m["quality"]["metrics"]["json_valid_pct"],
)
print("FP8 speed (full test):", {t: m[t]["timing"]["examples_per_s"] for t in TASKS})

**Paste back:** everything this cell printed (numbers only).

## 7 · Flush Drive, then disconnect
Upload back (summary numbers only):
- `MyDrive/vizwiz/runs/e4-fp8-e5/metrics.json`
- `MyDrive/vizwiz/runs/e4-fp8-e5/compare_answer.json`, `compare_answerability.json`, `compare_quality.json`
- `MyDrive/vizwiz/runs/e4-speed/speed_memory.json`

Then Runtime → **Disconnect and delete runtime**.

In [ ]:
drive.flush_and_unmount()
print("Drive flushed. Safe to disconnect.")